In [1]:
import torch 
import soundfile as sf
from qwen_tts import Qwen3TTSModel 

SoX could not be found!

    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 


In [3]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"사용 디바이스: {device}")

사용 디바이스: cuda:0


# 2. 보이스 디자인

In [1]:
import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel
from IPython.display import Audio, display
import numpy as np


SoX could not be found!

    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 


In [2]:

device = "cuda:0" if torch.cuda.is_available() else "cpu"

model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",  # VoiceDesign 모델
    device_map=device,
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:


wavs1, sr = model.generate_voice_design(
    text="안녕하세요, 오늘 날씨가 정말 좋네요!",
    instruct="women, speack in a sad tone",  # 목소리 묘사
)

sf.write("voice_design_sad.wav", wavs1[0], sr)

wavs2, sr = model.generate_voice_design(
    text='안녕하세요. 오늘 날씨가 정말 좋네요!',
    instruct='30 years old, male, speak in a happy tone'
)

sf.write('voice_design_happy.wav', wavs2[0],sr)


display(Audio("voice_design_sad.wav", autoplay=True))
display(Audio("voice_design_happy.wav", autoplay=True))

KeyboardInterrupt: 

## 2) 루프로 여러 문장 만들기

In [10]:
#볼륨 정규화
def normalize_volume(audio, target_db=-20.0):
    """RMS 기준으로 목표 dB에 맞게 볼륨 정규화"""
    rms = np.sqrt(np.mean(audio ** 2))
    if rms == 0:
        return audio
    target_rms = 10 ** (target_db / 20)
    gain = target_rms / rms
    # 클리핑 방지 (-1 ~ 1 범위 초과 시 제한)
    normalized = audio * gain
    return np.clip(normalized, -1.0, 1.0)

In [18]:
# 2. 문장별 성별, 연령, 감정이 포함된 스크립트 정의
fairy_tale_script = [
    {
        "text": "옛날 옛적에 한 나무꾼이 살고 있었단다.",
        "instruct": "70-year-old grandmother, low warm voice, gentle storytelling tone"
    },
    {
        "text": "와! 저기 좀 봐! 커다란 황금 도끼가 있어!",
        "instruct": "high-pitched 10-year-old girl, energetic"
    },
    {
        "text": "허허허, 네가 아주 정직한 아이로구나.",
        "instruct": "50-yesr-old senior women, low voice, slowly"
    }
]
all_audio_segments = []
final_sr = 24000 # 기본 샘플 레이트 예비 설정

PAUSE_DURATION = 0.8  # 문장 사이 무음 길이 (초) — 취향껏 조절
TARGET_DB = -20.0       #목표 볼륨 (dB)

results = {}
for i, line in enumerate(fairy_tale_script):
    results[i] = model.generate_voice_design(
        text=line["text"],
        language="korean",
        instruct=line["instruct"]
    )

# 합치기
final_sr = results[0][1]
all_audio_segments = []

for i in range(len(results)):
    wavs, sr = results[i]
    audio = wavs[0]
    

    normalized = normalize_volume(audio, target_db=TARGET_DB)  # 볼륨 맞추기
    rms = np.sqrt(np.mean(normalized ** 2))
    db = 20 * np.log10(rms) if rms > 0 else -np.inf
    print(f"세그먼트 {i} - RMS dB: {db:.2f}")
    
    all_audio_segments.append(normalized)
    
    if i < len(results) - 1:
        silence = np.zeros(int(final_sr * PAUSE_DURATION))
        all_audio_segments.append(silence)

combined = np.concatenate(all_audio_segments)
sf.write('./audios/fairy_tale_combined.wav', combined, final_sr)
print(f"저장 완료! 총 길이: {len(combined)/final_sr:.2f}초")


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


KeyboardInterrupt: 

# TTS

## 1) 모델 불러오기

In [4]:
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice",  # 가장 가벼운 모델부터
    device_map=device,
    dtype=torch.bfloat16,
    attn_implementation="sdpa",  # Windows는 sdpa 고정
)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
import soundfile as sf
# 한국어 테스트
wavs, sr = model.generate_custom_voice(
    text="안녕하세요, 저는 인공지능 스피커입니다.",
    language="Korean",
    speaker="sohee",  # 목록 확인 후 한국어 지원 화자로 변경
)


sf.write("./audios/qwen3_test.wav", wavs[0], sr)
print("✅ test.wav 생성 완료!")

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


✅ test.wav 생성 완료!


# 보이스 클론

In [3]:
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-0.6B-Base",  # Base 모델
    device_map=device,
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [4]:

import time


ref_audio = "./audios/my_voice.mp3"
ref_text  = "안녕하세요. 개그우먼 이수지입니다. 반갑습니다."
emotion_label = "Scary" # 비전 모델에서 넘어온 값
start_time = time.time()
wavs, sr = model.generate_voice_clone(
    text = "이제 그만 책을 읽어요",
    language="Korean",
    ref_audio=ref_audio,
    x_vector_only_mode=True
)
end_time = time.time()
print(f"GPU 생성 소요 시간: {end_time - start_time:.2f}초")

sf.write("./audios/output_voice_clone.wav", wavs[0], sr)


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


GPU 생성 소요 시간: 10.90초


In [ ]:

import time

ref_audio = "./audios/my_voice.mp3"
ref_text  = "안녕하세요. 개그우먼 이수지입니다. 반갑습니다."

start_time = time.time()
wavs, sr = model.generate_voice_clone(
    text = "(whispering) 비밀인데 사실은...",
    language="Korean",
    ref_audio=ref_audio,
    x_vector_only_mode=True
)
end_time = time.time()
print(f"GPU 생성 소요 시간: {end_time - start_time:.2f}초")

sf.write("./audios/output_voice_clone.wav", wavs[0], sr)


In [ ]:
#cpu로 돌려보기
# 모델이 이미 생성되어 있다면
model.to("cpu")

# 이후 생성 코드 실행
ref_audio = "./audios/my_voice.mp3"
ref_text  = "안녕하세요. 개그우먼 이수지입니다. 반갑습니다."

# 시간 측정을 위해 라이브러리 추가
import time
start_time = time.time()

wavs, sr = model.generate_voice_clone(
    text = "떡 하나주면 안 잡아먹지!!!! 으르렁!!!!",
    language="Korean",
    ref_audio=ref_audio,
    ref_text=ref_text,
)

end_time = time.time()
print(f"CPU 생성 소요 시간: {end_time - start_time:.2f}초")

sf.write("./audios/output_voice_clone.wav", wavs[0], sr)

=> 아쉽게도 지시문을 text에 끼워넣는 방식은 모두 실패

# Base 모델 뜯어보기

## 1) 보이스 클론 함수 뜯어보기

In [20]:
# generate_voice_clone 소스 확인
import inspect
print(inspect.getsource(model.generate_voice_clone))

#input_texts = [self._build_assistant_text(t) for t in texts]
# talker_codes_list, _ = self.model.generate(
#     input_ids=input_ids,
#     ref_ids=ref_ids,
#     voice_clone_prompt=voice_clone_prompt_dict,
#     languages=languages,
#     non_streaming_mode=non_streaming_mode,
#     **gen_kwargs,
# )


    @torch.no_grad()
    def generate_voice_clone(
        self,
        text: Union[str, List[str]],
        language: Union[str, List[str]] = None,
        ref_audio: Optional[Union[AudioLike, List[AudioLike]]] = None,
        ref_text: Optional[Union[str, List[Optional[str]]]] = None,
        x_vector_only_mode: Union[bool, List[bool]] = False,
        voice_clone_prompt: Optional[Union[Dict[str, Any], List[VoiceClonePromptItem]]] = None,
        non_streaming_mode: bool = False,
        **kwargs,
    ) -> Tuple[List[np.ndarray], int]:
        """
        Voice clone speech using the Base model.

        You can provide either:
          - (ref_audio, ref_text, x_vector_only_mode) and let this method build the prompt, OR
          - `VoiceClonePromptItem` returned by `create_voice_clone_prompt`, OR
          - a list of `VoiceClonePromptItem` returned by `create_voice_clone_prompt`.
        
        `ref_audio` Supported forms:
        - str: wav path / URL / base64 audio string
   

In [21]:
print(inspect.getsource(model._build_assistant_text))

    def _build_assistant_text(self, text: str) -> str:
        return f"<|im_start|>assistant\n{text}<|im_end|>\n<|im_start|>assistant\n"



In [22]:
print(inspect.getsource(model._build_ref_text))


    def _build_ref_text(self, text: str) -> str:
        return f"<|im_start|>assistant\n{text}<|im_end|>\n"



In [23]:
def patched_build_assistant_text(text: str, instruction: str = "") -> str:
    if instruction:
        return (
            f"<|im_start|>user\n{instruction}<|im_end|>\n"
            f"<|im_start|>assistant\n{text}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
    else:
        return f"<|im_start|>assistant\n{text}<|im_end|>\n<|im_start|>assistant\n"

# 원하는 감정을 instruction에 넣어서 호출
import types

emotion = "Speak in a sad tone"

model._build_assistant_text = lambda text: patched_build_assistant_text(text, instruction=emotion)

# 그냥 평소처럼 호출하면 됨
wavs, sr = model.generate_voice_clone(
    text="안녕하세요! 오늘 정말 좋은 날이에요!",
    language="Korean",
    ref_audio=ref_audio,
    ref_text=ref_text,
)
sf.write("./audios/output_voice_clone.wav", wavs[0], sr)


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


## 2) model.generate() 직접 호출

In [5]:
import torch
import soundfile as sf

# ── 1. voice_clone_prompt 생성 ──────────────────────────────
prompt_items = model.create_voice_clone_prompt(
    ref_audio=ref_audio,
    ref_text=ref_text,
)
voice_clone_prompt_dict = model._prompt_items_to_voice_clone_prompt(prompt_items)

# ── 2. instruction 포함한 input_ids 직접 조립 ────────────────
instruction = "Speak in a sad, emotional tone"
text = "안녕하세요! 오늘 정말 좋은 날이에요!"

# user 턴에 instruction, assistant 턴에 text
full_text = (
    f"<|im_start|>user\n{instruction}<|im_end|>\n"
    f"<|im_start|>assistant\n{text}<|im_end|>\n"
    f"<|im_start|>assistant\n"
)
input_ids = model._tokenize_texts([full_text])

# ── 3. ref_ids 조립 ─────────────────────────────────────────
ref_ids = []
for it in prompt_items:
    if it.ref_text:
        ref_tok = model._tokenize_texts([model._build_ref_text(it.ref_text)])[0]
        ref_ids.append(ref_tok)
    else:
        ref_ids.append(None)

# ── 4. generate 직접 호출 ────────────────────────────────────
gen_kwargs = model._merge_generate_kwargs()

talker_codes_list, _ = model.model.generate(
    input_ids=input_ids,
    ref_ids=ref_ids,
    voice_clone_prompt=voice_clone_prompt_dict,
    languages=["Korean"],
    non_streaming_mode=False,
    **gen_kwargs,
)

# ── 5. 디코딩 (generate_voice_clone 소스 그대로) ─────────────
codes_for_decode = []
for i, codes in enumerate(talker_codes_list):
    ref_code_list = voice_clone_prompt_dict.get("ref_code", None)
    if ref_code_list is not None and ref_code_list[i] is not None:
        codes_for_decode.append(
            torch.cat([ref_code_list[i].to(codes.device), codes], dim=0)
        )
    else:
        codes_for_decode.append(codes)

wavs_all, fs = model.model.speech_tokenizer.decode(
    [{"audio_codes": c} for c in codes_for_decode]
)

# ── 6. ref 부분 잘라내고 저장 ────────────────────────────────
wavs_out = []
for i, wav in enumerate(wavs_all):
    ref_code_list = voice_clone_prompt_dict.get("ref_code", None)
    if ref_code_list is not None and ref_code_list[i] is not None:
        ref_len = int(ref_code_list[i].shape[0])
        total_len = int(codes_for_decode[i].shape[0])
        cut = int(ref_len / max(total_len, 1) * wav.shape[0])
        wavs_out.append(wav[cut:])
    else:
        wavs_out.append(wav)

sf.write("./audios/output_direct.wav", wavs_out[0], fs)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
